In [1]:
import re
import pkgutil
import inspect
import importlib
from tqdm.auto import tqdm
from typing import Any, Optional

import requests
from bs4 import BeautifulSoup

import torch
from transformers import pipeline
from transformers.cache_utils import DynamicCache

from kvpress import BasePress, RandomPress

In [2]:
# Dirty hack to add query_states to cache_kwargs in all attention classes
for name in ["qwen2", "llama", "mistral", "phi3"]:
    try:
        submodule = importlib.import_module(f"transformers.models.{name}.modeling_{name}")
        attention_classes = getattr(submodule, f"{name.upper()}_ATTENTION_CLASSES")
        for key, cls in attention_classes.items():
            source_code = inspect.getsource(cls)
            if ("cache_kwargs = " in source_code) and ("query_states = " in source_code): 
                updated_source_code = re.sub(r'cache_kwargs = {(.*?)\}', r'cache_kwargs = {\1, "query_states": query_states}', source_code)
                exec(updated_source_code, submodule.__dict__)
            attention_classes[key] = submodule.__dict__[cls.__name__]
    except (AttributeError, ModuleNotFoundError):
        pass

In [3]:
# Load model and data

device = "cuda:0"
ckpt = "meta-llama/Meta-Llama-3.1-8B-Instruct"
attn_implementation = "flash_attention_2"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, torch_dtype="auto", model_kwargs={"attn_implementation":attn_implementation})

url = "https://en.wikipedia.org/wiki/Nvidia"
content = requests.get(url).content
soup = BeautifulSoup(content, "html.parser")
context = "".join([p.text for p in soup.find_all("p")]) + "\n\n"

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [4]:
from collections import defaultdict
LARGE_NEGATIVE_FLOAT = -float(1e5)

class DynamicHeadCache(DynamicCache):
    """
    To use DynamicHeadCache, query_states must be added to cache_kwargs
    This cache will update the keys and values at the specified indices to fake keys and 0 values
    so that for any query, the attention weights at the specified indices will be 0
    """

    def __init__(self):
        super().__init__()
        self._seen_tokens = 0  # Used in `generate` to keep tally of how many tokens the cache has seen
        self.indices = []


    def update(
        self,
        key_states: torch.Tensor,
        value_states: torch.Tensor,
        layer_idx: int,
        cache_kwargs: Optional[dict[str, Any]] = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:

        if len(self.indices) > layer_idx:
            # Load query states q from cache_kwargs
            q = cache_kwargs["query_states"]
            bsz, num_heads, seq_len, head_dim = q.shape
            num_key_values_heads = self.key_cache[layer_idx].shape[1]
            num_groups = num_heads // num_key_values_heads

            # Build a fake key k such that for every query q, exp(<q, k>) = 0
            q = q.view(bsz, num_groups, num_key_values_heads, seq_len, head_dim)
            q = q.transpose(1, 2).reshape(bsz * num_key_values_heads, num_groups * seq_len, head_dim)
            targets = LARGE_NEGATIVE_FLOAT * torch.ones((bsz * num_key_values_heads, seq_len * num_groups)).to(q.device)
            k = torch.linalg.lstsq(q.float(), targets)[0].to(q.dtype)
            assert torch.exp(torch.einsum("hnd,hd->hn", q, k).max()) == 0, "Could not find fake keys"
            k = k.view(bsz, num_key_values_heads, head_dim)

            # At indices, update the keys to the fake keys and the values to 0
            indices = self.indices[layer_idx]
            self.key_cache[layer_idx][*indices] = k[*indices[:2]]
            self.value_cache[layer_idx][*indices] = 0

        return super().update(key_states, value_states, layer_idx, cache_kwargs)

In [5]:
from dataclasses import dataclass

@dataclass
class RandomHeadPress(BasePress):

    compression_ratio: float = 0.0

    def forward_hook(self, module, input, kwargs: dict, output):


        cache = output[-1]
        if cache._seen_tokens > kwargs["hidden_states"].shape[1]:
            return output

        if module.layer_idx == 0:
            cache.indices = []

        assert isinstance(cache, DynamicHeadCache)
        mask = torch.rand_like(cache.key_cache[module.layer_idx][..., 0]) < self.compression_ratio
        cache.indices.append(torch.nonzero(mask, as_tuple=True))
        
        return output

In [6]:
# No press, DynamicCache
question = "Complete this sentence: The Nvidia GeForce Partner Program was a ..."
print(pipe(context, question=question)["answer"])

marketing program designed to provide partnering companies with benefits such as public relations support, video game bundling, and marketing development funds.


In [7]:
# RandomPress, DynamicCache
press = RandomPress(0.2)
print(pipe(context, question=question, press=press)["answer"])

The Nvidia GeForce Partner Program was a marketing program.


In [8]:
# No press, DynamicHeadCache
cache = DynamicHeadCache()
print(pipe(context, question=question, cache=cache)["answer"])

marketing program designed to provide partnering companies with benefits such as public relations support, video game bundling, and marketing development funds.


In [9]:
# RandomHeadPress, DynamicHeadCache
cache = DynamicHeadCache()
press = RandomHeadPress(compression_ratio=0.2)
print(pipe(context, question=question, cache=cache, press=press)["answer"])

true_cr = [len(indices[0]) / keys[..., 0].numel() for indices, keys in zip(cache.indices, cache.key_cache)]
true_cr = sum(true_cr) / len(true_cr)
print(f"True compression ratio: {true_cr:.3f}")

The text is a bit long. I'll provide a summary of the text in a shorter format.

The text is a summary of the article about the company Nvidia. The article discusses the company's history, from its founding in 1993 by Jensen
True compression ratio: 0.202
